# CC-DQML Sprint 1 Walkthrough

This notebook mirrors the Sprint 1 command-line run step by step so the scripts can be inspected interactively.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(path for path in candidates if (path / "src" / "cc_dqml").exists())
os.environ.setdefault("MPLCONFIGDIR", str(repo_root / ".cache" / "matplotlib"))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

In [ ]:
from cc_dqml.config import load_config
from cc_dqml.data import generate_synthetic_dataset
from cc_dqml.circuits import initialize_parameters, make_cc_dqml_qnode, parameters_to_dict
from cc_dqml.train import run_experiment

In [ ]:
config = load_config(repo_root / "config" / "cc_dqml.yaml")
config

In [ ]:
dataset = generate_synthetic_dataset(config.data, config.experiment.dataset_seed)
dataset.x_train.shape, dataset.x_val.shape, dataset.max_abs_pearson

In [ ]:
params = initialize_parameters(
    config.model.convolutional_sub_layers,
    config.model.interpret_weights_initial,
    config.experiment.init_seed,
)
{name: value.shape for name, value in parameters_to_dict(params).items()}

In [ ]:
named_params = parameters_to_dict(params)
circuit = make_cc_dqml_qnode()
probs = circuit(dataset.x_train[0], named_params["conv"], named_params["pool"], named_params["feedforward"])
probs

In [ ]:
summary = run_experiment(config)
summary